## 第8章 错误和异常

- **异常**
    - 定义：计算机正常处理中的中断，通常由错误条件引起，可以由程序的另一部分处理。
    - 阅读异常信息：异常信息中包含所发生错误的详细信息、错误发生的行，以及整个调用栈，**通常从下往上读**，找到第一个异常发生的地方。

- **捕获异常：**
    - 原则：Python中，先执行出问题再处理(**EAFP**)，其他语言中先执行检查再执行(**LBYL**)。
    - `try`语句：可能出错的代码块。
    - `except`语句：捕获异常，执行异常处理代码。
    - `else`语句：没有异常发生时执行。
    - `finally`语句：无论是否发生异常都执行。

In [ ]:
# 异常处理语法模板

try:                                # 可能会出错的代码
    1 / 0
except ZeroDivisionError:           # 处理特定异常
    print("除数不能为0")
except (ValueError, TypeError):     # 处理多个异常
    print("输入的不是数字")
except Exception as e:              # 处理未知异常(慎用，吞噬大部分异常)
    print(f"发生未知错误：{e}")
else:                               # 没有异常发生时执行
    print("没有异常发生")
finally:                            # 无论是否发生异常都执行
    print("清理资源")

- **抛出异常：**
    - `raise`语句：手动抛出异常。抛出异常会导致函数立即退出，类似`return`语句。

- **异常链：**
    - 抛出异常时指定来源：`raise Exception("错误信息") from e`。
    - `raise e from None`：显式地禁用异常链。

- **自定义异常：**
    - 自定义异常**必须继承`Exception`**(或其子类)，**绝不继承`BaseException`**！

- **异常继承关系：**

```text
BaseException                          # 所有异常基类
├── SystemExit                         # sys.exit()引发
├── KeyboardInterrupt                  # CTRL-C中断 ⚠️
├── GeneratorExit                      # 生成器关闭
└── Exception                          # 所有常规异常基类 ⭐
    ├── ArithmeticError                # 算术错误基类
    │   ├── OverflowError              # 结果过大
    │   ├── ZeroDivisionError          # 除以零
    │   └── FloatingPointError         # 浮点错误
    ├── LookupError                    # 查找错误基类
    │   ├── IndexError                 # 索引越界
    │   └── KeyError                   # 字典键不存在
    ├── ValueError                     # 值错误但类型正确
    ├── TypeError                      # 对象类型错误
    ├── AttributeError                 # 访问不存在的属性
    ├── NameError                      # 名称未找到
    │   └── UnboundLocalError          # 局部变量未绑定
    ├── RuntimeError                   # 运行时错误
    │   └── RecursionError             # 递归过深
    ├── NotImplementedError            # 功能未实现
    ├── StopIteration                  # 迭代器耗尽
    ├── ImportError                    # 导入失败
    │   └── ModuleNotFoundError        # 模块未找到
    ├── OSError                        # 操作系统错误
    │   ├── FileNotFoundError          # 文件不存在
    │   ├── PermissionError            # 权限不足
    │   └── IsADirectoryError          # 是目录而非文件
    ├── UnicodeError                   # Unicode相关错误
    │   ├── UnicodeDecodeError
    │   └── UnicodeEncodeError
    ├── MemoryError                    # 内存不足
    └── BufferError                    # 缓冲区错误

- **核心知识脉络:**

```text
错误与异常
│
├── 1. Python中的异常⭐
│   ├── 异常 = 正常处理的中断，可被程序其他部分处理
│   └── 正确测试 = "对代码做可怕的事情"
│
├── 2. 阅读Traceback⭐
│   ├── 从下往上读！最后一行最重要
│   ├── 最后一行 = 错误类型 + 原因
│   ├── 中间行 = 错误发生位置（文件、行号、函数）
│   └── 上方行 = 调用栈回溯
│
├── 3. LBYL vs EAFP ⭐⭐⭐
│   ├── LBYL：先检查再执行（不推荐）
│   │   ├── 快乐路径两次处理
│   │   └── 有竞争条件风险
│   ├── EAFP：先执行再处理异常（推荐⭐⭐⭐）
│   │   ├── 快乐路径一次处理，更高效
│   │   ├── 无竞争条件
│   │   └── 思维负担更轻
│   └── try/except 是EAFP的实现工具
│
├── 4. 多个异常⭐
│   ├── 元组语法：except (Type1, Type2):
│   └── 尿布反模式⚠️⚠️⚠️
│       ├── 裸except: 吞噬所有异常+丢弃traceback
│       ├── 程序在无效状态继续→奇怪行为
│       ├── 无法CTRL-C退出（KeyboardInterrupt被捕获）
│       └── except Exception: 仍是变体，唯一可接受场景是配合logging
│
├── 5. 主动抛出异常（raise）⭐
│   ├── raise ExceptionType("message")
│   ├── raise 立即跳出函数（类似return）
│   └── 捕获自己抛出的异常
│
├── 6. 异常用于控制流⭐
│   ├── 异常是对象，可用as e绑定，用str(e)提取信息
│   └── 字典查找：try/except KeyError（EAFP优势）
│
├── 7. Logging与异常⭐
│   ├── logging.basicConfig(filename, level)
│   ├── 五个级别：DEBUG < INFO < WARNING < ERROR < CRITICAL
│   ├── logging.info(e)：仅记录消息（无traceback）
│   ├── logging.exception(e)：记录ERROR+完整traceback ⭐
│   └──  ⚠️ basicConfig()只在if __name__ == "__main__":中
│
├── 8. 冒泡⭐
│   ├── 未捕获异常应记录ERROR并让程序崩溃
│   ├── except Exception as e: logging.exception(e); raise ⭐
│   │   ├── 只捕获Exception子类
│   │   └── 不捕获KeyboardInterrupt/StopIteration
│   └── 异常链：raise NewError() from e ⭐⭐
│       ├── 保留原始异常作为__cause__
│       └── 调试价值极高
│
├── 9. else和finally ⭐⭐
│   ├── else子句："如果一切顺利"
│   │   ├── try无异常 → 执行else
│   │   └── try有异常 → 不执行else
│   └── finally子句："之后一切"⭐⭐
│       ├── 无论有无异常、有无return/raise都执行
│       ├── finally在return/raise之前执行⚠️
│       └── 适合清理资源：关闭文件、释放锁
│
└── 10. 自定义异常⭐
    ├── 继承Exception（绝不继承BaseException）⚠️
    ├── 三个标准至少符合两个才值得自定义
    │   1. 现有异常无法描述
    │   2. 会多次引发/捕获
    │   3. 需单独捕获
    └── __init__可选但推荐，可设默认消息
```

- **警告与提示表**

| 类型 | 内容                                                         |
| ---- | ------------------------------------------------------------ |
| ⚠️    | 阅读traceback**从下往上**，最后一行最重要（错误类型+原因），修复前先完整理解 |
| ⚠️    | 裸`except:`是**尿布反模式**——吞噬所有异常+丢弃traceback，程序在无效状态继续 ⚠️⚠️ |
| ⚠️    | 裸`except:`会捕获`KeyboardInterrupt`，导致程序**无法用CTRL-C退出** ⚠️⚠️ |
| ⚠️    | `except Exception:`仍是尿布变体，唯一可接受场景是配合`logging`+`raise`冒泡 |
| ⚠️    | `finally`子句在`return`/`raise`**之前执行**，即使源代码顺序相反 ⚠️ |
| ⚠️    | 自定义异常**必须继承`Exception`**，绝不继承`BaseException` ⚠️ |
| ⚠️    | `logging.basicConfig()`应只放在`if __name__ == "__main__":`中，改变全局行为 ⚠️ |
| ⚠️    | `raise ... from None`会丢失调试信息，不建议使用              |
| 💡    | EAFP是Python首选错误处理哲学：先尝试，出问题再处理，快乐路径更高效 ⭐⭐⭐ |
| 💡    | 捕获异常用`as e`获取异常对象，`str(e)`可提取关键信息         |
| 💡    | 多种异常用元组：`except (ValueError, UnicodeError):` 统一处理 |
| 💡    | 冒泡模式：`except Exception as e: logging.exception(e); raise` ⭐ |
| 💡    | `raise NewError() from e`保留异常链，调试时能追溯完整因果链 ⭐⭐ |
| 💡    | `finally`子句适合放清理逻辑：关闭文件、释放资源，必执行 ⭐⭐   |
| 💡    | `else`子句让"无异常时的逻辑"与"异常处理"分离，代码更清晰     |
| 💡    | 自定义异常三个标准至少符合两个：描述不足、多次使用、需单独捕获 |
| 💡    | 捕获`KeyboardInterrupt`和`EOFError`处理程序退出，打印友好消息后`sys.exit(0)` |
| 💡    | 字典查找优先用`try/except KeyError`而非`if key in d`（EAFP优势，避免竞争条件） |